# CSSE4011 Machine Learning and Edge AI Workshop

### 0. Connect to a runtime and change the runtime type to **"T4 GPU"**

Click the down-arrow button in the top-right corner, as shown in the screenshots below.

> **Important:** Please make sure you select **T4 GPU**, as using a different runtime may lead to much longer execution times.

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/change_runtime_type.png?raw=true" width="300"> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/runtime_type.png?raw=true" width="300">

## 1. Install Required Packages and Upload MNIST Data

In [ ]:
!git clone https://github.com/YangLi309/CSSE4011-ML-Workshop.git
%cd CSSE4011-ML-Workshop/

In [ ]:
# Install ONNX for model export
!pip3 install onnx onnxscript onnxruntime-gpu onnxconverter-common


## 2. Introduction

### What is PyTorch?
PyTorch is an open-source deep learning framework known for its flexibility and ease of use, making it ideal for rapid prototyping and research.

### What is ONNX?
ONNX (Open Neural Network Exchange) is an open standard format for representing machine learning models, enabling interoperability between various frameworks and devices.

### Why are these important for Edge AI?
- **PyTorch**: Facilitates easy model development and training.
- **ONNX**: Allows exporting models to a standardized format for deployment across different platforms, including resource-constrained edge devices.

### What You Will Learn in This Workshop
- Load and adapt a pretrained model for a new task.
- Evaluate model performance.
- Prune the model to improve inference time.
- Export the model to ONNX format.

## 3. Load Pretrained AlexNet and Adapte the Model for MNIST 10

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import alexnet, AlexNet_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Pretrained AlexNet with the final layer replaced for MNIST (10 classes)
model = alexnet(weights=AlexNet_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)  # For 10 classes

model = model.to(device)
print("Pretrained AlexNet model is loaded.")

## 4. Evaluate the Pretrained AlexNet on MNIST

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Transform for MNIST images (which are 28x28 grayscale)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load MNIST test data
val_dataset = ImageFolder(root='./data/val', transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_dataset = ImageFolder(root='./data/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Evaluate the model (with modified classifier but before fine-tuning)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (before finetuning) on MNIST test data: {100 * correct / total:.2f}%")
print("Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

## 5. Freeze All Layers Except Classifier

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/alexnet.png?raw=true" width="500">

In [ ]:
# Transfer learning: keep ImageNet features fixed and only train the new MNIST head.
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

## 6. Load MNIST Train Dataset

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming your dataset is stored in "./data/" with subfolders 0/, 1/, ..., 9/
train_dataset = ImageFolder(root='./data/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 7. Finetune the Classifier on MNIST

### Training epochs and corresponding model accuracy

- **1 epoch:** 96.45% accuracy  
- **5 epochs:** 97.02% accuracy  
- **10 epochs:** 99.30% accuracy  

> **Note:** The default setting is **1 epoch** for time efficiency, but you may increase the number of epochs to achieve better performance.

> **Reference:** Training for **1 epoch** should take approximately **3 minutes**. If it takes significantly longer, check whether your runtime type is set to **CPU** instead of **GPU**.

In [ ]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# You can increase the number of epochs if you want to have higher accuracy
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Wrap train_loader with tqdm for progress bar
    train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for images, labels in train_loader_tqdm:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        train_loader_tqdm.set_postfix({
            'loss': running_loss/total,
            'acc': 100.*correct/total
        })

    # Print epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f'\nEpoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

## 8. Evaluate the Finetuned Model

In [ ]:
# Evaluate the finetuned model on the test set (same loop structure as pre-training eval).
from tqdm import tqdm

correct = 0
total = 0
all_preds = []
all_labels = []
finetuned_model = model
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Evaluating'):
        images, labels = images.to(device), labels.to(device)
        outputs = finetuned_model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (after finetuning) on MNIST test data: {100 * correct / total:.2f}%")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

## 9. Model Export to ONNX
> We export the model to ONNX to make it easier to deploy and run across different inference backends beyond PyTorch. It also enables runtime-specific optimizations such as FP16 or INT8 acceleration with tools like ONNX Runtime or TensorRT.

In [ ]:
import torch
import onnx

# Paths
onnx_path = "../alexnet_mnist.onnx"

# Export on CPU for simplicity and compatibility
export_model = finetuned_model.cpu().eval()

# Example input for export
# Keep this shape consistent with your model's real expected input
dummy_input = (torch.randn(1, 3, 224, 224),)

# Export PyTorch model to ONNX
# dynamic_shapes makes batch dimension flexible
onnx_program = torch.onnx.export(
    export_model,
    dummy_input,
    dynamo=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_shapes={"x": {0: "batch"}},   # use your forward arg name, likely x
)

# Save ONNX file
onnx_program.save(onnx_path)
print(f"Saved ONNX model to: {onnx_path}")

# Optional sanity check
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model check passed.")

## 10. Verify that the exported ONNX model matches its PyTorch counterpart (same inputs, same logits within tolerance).

In [ ]:
# Same random input for both checks: PyTorch logits should match ONNXRuntime for each exported model.
import onnxruntime as ort
import numpy as np

def verify_onnx(torch_model, onnx_path, x, label):
    torch_model.eval()
    with torch.no_grad():
        torch_out = torch_model(x).cpu().numpy()
    sess = ort.InferenceSession(onnx_path)
    in_name = sess.get_inputs()[0].name
    onnx_out = sess.run(None, {in_name: x.cpu().numpy()})[0]
    ok = np.allclose(torch_out, onnx_out, atol=1e-4)
    mad = float(np.max(np.abs(torch_out - onnx_out)))
    print(f"{label}: match={ok}, max abs diff={mad:.6e}")


device = next(finetuned_model.parameters()).device
dummy_input = torch.randn(1, 3, 224, 224, device=device)

verify_onnx(finetuned_model, "../alexnet_mnist.onnx", dummy_input, "Finetuned")

## 11. Convert FP32 to FP16
> Converting a model from FP32 to FP16 reduces precision and model size, which can improve inference speed on supported hardware, especially GPUs.

In [ ]:
import os
import onnx
from onnxconverter_common import float16

onnx_path = "../alexnet_mnist.onnx"
fp16_onnx_path = "../alexnet_mnist_fp16.onnx"

# Load FP32 ONNX model
model = onnx.load(onnx_path)

# Convert to FP16
# keep_io_types=False means model inputs/outputs will also become float16
# disable_shape_infer=True can help if shape inference is fragile on your graph
model_fp16 = float16.convert_float_to_float16(
    model,
    keep_io_types=False,
    disable_shape_infer=True,
)

# Save FP16 ONNX model
onnx.save(model_fp16, fp16_onnx_path)

print(f"Saved FP16 ONNX model to: {fp16_onnx_path}")

fp32_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
fp16_size_mb = os.path.getsize(fp16_onnx_path) / (1024 * 1024)

print(f"FP32 ONNX size: {fp32_size_mb:.2f} MB")
print(f"FP16 ONNX size: {fp16_size_mb:.2f} MB")
print(f"Size reduction: {(fp32_size_mb - fp16_size_mb) / fp32_size_mb * 100:.2f}%")

## 12. INT8 Quantization
> Static INT8 quantization with calibration data by first optionally preprocessing the FP32 ONNX model, then using a small subset of training images to estimate activation ranges.

In [ ]:
import os
import numpy as np
import onnx
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

from onnxruntime.quantization import (
    quant_pre_process,
    quantize_static,
    CalibrationDataReader,
    QuantFormat,
    QuantType,
)

# Paths
fp32_onnx_path = "../alexnet_mnist.onnx"
preprocessed_onnx_path = "../alexnet_mnist_preprocessed.onnx"
int8_onnx_path = "../alexnet_mnist_int8_static.onnx"

# Calibration dataset root
calib_data_dir = "./data/train"

# Use a small subset for calibration to keep it fast
calib_subset_size = 512
calib_batch_size = 1


def build_calib_loader(data_root: str, batch_size: int = 1, subset_size: int | None = None):
    # Match the same preprocessing used for model inference
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = ImageFolder(root=data_root, transform=transform)

    if subset_size is not None:
        subset_size = min(subset_size, len(dataset))
        dataset = Subset(dataset, range(subset_size))

    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)


class ImageCalibrationDataReader(CalibrationDataReader):
    """
    ONNX Runtime calls get_next() repeatedly to fetch calibration samples.
    Each returned dict must map ONNX input name -> NumPy input batch.
    """
    def __init__(self, dataloader, input_name="input", input_dtype=np.float32):
        self.dataloader = dataloader
        self.input_name = input_name
        self.input_dtype = input_dtype
        self._iterator = iter(self.dataloader)

    def get_next(self):
        try:
            images, _ = next(self._iterator)
            images_np = images.numpy().astype(self.input_dtype)
            return {self.input_name: images_np}
        except StopIteration:
            return None

    def rewind(self):
        self._iterator = iter(self.dataloader)


# Optional preprocessing step before quantization
# In practice this can help, but for some graphs symbolic shape inference can fail.
# If that happens in your environment, skip this block and quantize fp32_onnx_path directly.
print("Running ONNX pre-processing...")
quant_pre_process(
    input_model=fp32_onnx_path,
    output_model_path=preprocessed_onnx_path,
    skip_symbolic_shape=True,
)
print(f"Saved preprocessed ONNX model to: {preprocessed_onnx_path}")

# Build calibration loader
calib_loader = build_calib_loader(
    calib_data_dir,
    batch_size=calib_batch_size,
    subset_size=calib_subset_size,
)

# The ONNX input name should match the exported model input name.
# If your model uses a different input name, replace "input" below.
data_reader = ImageCalibrationDataReader(
    dataloader=calib_loader,
    input_name="input",
    input_dtype=np.float32,
)

print("Running static INT8 quantization with calibration data...")
quantize_static(
    model_input=preprocessed_onnx_path,
    model_output=int8_onnx_path,
    calibration_data_reader=data_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
)

print(f"Saved INT8 quantized ONNX model to: {int8_onnx_path}")

# Size comparison
fp32_size_mb = os.path.getsize(fp32_onnx_path) / (1024 * 1024)
int8_size_mb = os.path.getsize(int8_onnx_path) / (1024 * 1024)

print(f"FP32 ONNX size: {fp32_size_mb:.2f} MB")
print(f"INT8 ONNX size: {int8_size_mb:.2f} MB")
print(f"Size reduction: {(fp32_size_mb - int8_size_mb) / fp32_size_mb * 100:.2f}%")

## 13. Compare Native Pytorch, ONNX FP32, ONNX FP16 and ONNX INT8
> Compare the original native PyTorch model with ONNX FP32, ONNX FP16, and ONNX INT8 versions to evaluate differences in accuracy, inference speed, and model size on GPU and CPU, respectively.

In [ ]:
import os
import time
import numpy as np
import torch
import onnxruntime as ort
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# =========================================================
# Config
# =========================================================
fp32_onnx_model = "../alexnet_mnist.onnx"
fp16_onnx_model = "../alexnet_mnist_fp16.onnx"
int8_onnx_model = "../alexnet_mnist_int8_static.onnx"

val_data_dir = "./data/val"
batch_size = 1   # keep 1 if your ONNX models use fixed batch size

# =========================================================
# Data loader
# =========================================================
def build_loader(data_root: str, batch_size: int = 1):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    dataset = ImageFolder(root=data_root, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
    )
    return dataset, loader


# =========================================================
# PyTorch evaluation
# =========================================================
def evaluate_pytorch_model(model, loader, device):
    model = model.to(device).eval()

    correct = 0
    total = 0
    latencies = []
    all_preds = []
    all_labels = []

    model_dtype = next(model.parameters()).dtype

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device=device, dtype=model_dtype)
            labels = labels.to(device)

            if device.type == "cuda":
                torch.cuda.synchronize()
            start_time = time.perf_counter()

            outputs = model(images)

            if device.type == "cuda":
                torch.cuda.synchronize()
            end_time = time.perf_counter()

            latencies.append(end_time - start_time)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = 100.0 * correct / total
    total_time = sum(latencies)
    fps = total / total_time if total_time > 0 else float("nan")

    return {
        "accuracy": accuracy,
        "fps": fps,
        "total_time": total_time,
        "preds": all_preds,
        "labels": all_labels,
        "providers": [str(device)],
    }


# =========================================================
# ONNX evaluation
# =========================================================
def evaluate_onnx_model(onnx_path, loader, provider_name, input_dtype=np.float32):
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    if provider_name == "cpu":
        providers = ["CPUExecutionProvider"]
    elif provider_name == "cuda":
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    else:
        raise ValueError(f"Unsupported provider_name: {provider_name}")

    session = ort.InferenceSession(
        onnx_path,
        sess_options=sess_opts,
        providers=providers,
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    correct = 0
    total = 0
    latencies = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images_np = images.numpy().astype(input_dtype)
        labels_np = labels.numpy()

        start_time = time.perf_counter()
        outputs = session.run([output_name], {input_name: images_np})[0]
        end_time = time.perf_counter()

        latencies.append(end_time - start_time)

        preds = np.argmax(outputs, axis=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels_np.tolist())

        correct += (preds == labels_np).sum()
        total += len(labels_np)

    accuracy = 100.0 * correct / total
    total_time = sum(latencies)
    fps = total / total_time if total_time > 0 else float("nan")

    return {
        "accuracy": accuracy,
        "fps": fps,
        "total_time": total_time,
        "preds": all_preds,
        "labels": all_labels,
        "providers": session.get_providers(),
    }


# =========================================================
# Helpers
# =========================================================
def get_pytorch_model_size_mb(model, file_name="tmp_finetuned_model.pth"):
    torch.save(model.state_dict(), file_name)
    size_mb = os.path.getsize(file_name) / (1024 * 1024)
    os.remove(file_name)
    return size_mb

def print_summary_table(title, results, model_sizes_mb):
    print(f"\n{title}")
    print(f"{'Model':<16} {'Accuracy (%)':>14} {'FPS':>12} {'Time (s)':>12} {'Size (MB)':>12}")
    print("-" * 74)

    order = ["PyTorch", "FP32 ONNX", "FP16 ONNX", "INT8 ONNX"]
    for model_name in order:
        r = results[model_name]
        size_mb = model_sizes_mb[model_name]
        print(
            f"{model_name:<16} "
            f"{r['accuracy']:>14.2f} "
            f"{r['fps']:>12.1f} "
            f"{r['total_time']:>12.2f} "
            f"{size_mb:>12.2f}"
        )

    print("-" * 74)

def print_sample_predictions(results, label="INT8 ONNX", max_items=10):
    preds = results[label]["preds"]
    labels = results[label]["labels"]

    print(f"\nSample predictions ({label}):")
    print(f"  {'True':>6}  {'Pred':>6}  {'Match':>6}")
    for true, pred in list(zip(labels, preds))[:max_items]:
        mark = "✓" if true == pred else "✗"
        print(f"  {true:>6}  {pred:>6}  {mark:>6}")


# =========================================================
# Main
# =========================================================
dataset, loader = build_loader(val_data_dir, batch_size=batch_size)
print(f"Validation set: {len(dataset)} images | batch size: {batch_size}\n")

model_sizes_mb = {
    "PyTorch": get_pytorch_model_size_mb(finetuned_model),
    "FP32 ONNX": os.path.getsize(fp32_onnx_model) / (1024 * 1024),
    "FP16 ONNX": os.path.getsize(fp16_onnx_model) / (1024 * 1024),
    "INT8 ONNX": os.path.getsize(int8_onnx_model) / (1024 * 1024),
}

# ---------------------------------------------------------
# CPU comparison
# ---------------------------------------------------------
cpu_results = {}

print("Running PyTorch on CPU ...")
cpu_results["PyTorch"] = evaluate_pytorch_model(finetuned_model, loader, torch.device("cpu"))
print(f"PyTorch providers: {cpu_results['PyTorch']['providers']}")

print("Running FP32 ONNX on CPU ...")
cpu_results["FP32 ONNX"] = evaluate_onnx_model(
    fp32_onnx_model, loader, provider_name="cpu", input_dtype=np.float32
)
print(f"FP32 ONNX providers: {cpu_results['FP32 ONNX']['providers']}")

print("Running FP16 ONNX on CPU ...")
cpu_results["FP16 ONNX"] = evaluate_onnx_model(
    fp16_onnx_model, loader, provider_name="cpu", input_dtype=np.float16
)
print(f"FP16 ONNX providers: {cpu_results['FP16 ONNX']['providers']}")

print("Running INT8 ONNX on CPU ...")
cpu_results["INT8 ONNX"] = evaluate_onnx_model(
    int8_onnx_model, loader, provider_name="cpu", input_dtype=np.float32
)
print(f"INT8 ONNX providers: {cpu_results['INT8 ONNX']['providers']}")

print_summary_table("CPU Summary", cpu_results, model_sizes_mb)
print_sample_predictions(cpu_results, label="INT8 ONNX")

# ---------------------------------------------------------
# CUDA comparison
# ---------------------------------------------------------
cuda_results = {}
cuda_available = torch.cuda.is_available() and ("CUDAExecutionProvider" in ort.get_available_providers())

if cuda_available:
    print("\nRunning PyTorch on CUDA ...")
    cuda_results["PyTorch"] = evaluate_pytorch_model(finetuned_model, loader, torch.device("cuda"))
    print(f"PyTorch providers: {cuda_results['PyTorch']['providers']}")

    print("Running FP32 ONNX on CUDA ...")
    cuda_results["FP32 ONNX"] = evaluate_onnx_model(
        fp32_onnx_model, loader, provider_name="cuda", input_dtype=np.float32
    )
    print(f"FP32 ONNX providers: {cuda_results['FP32 ONNX']['providers']}")

    print("Running FP16 ONNX on CUDA ...")
    cuda_results["FP16 ONNX"] = evaluate_onnx_model(
        fp16_onnx_model, loader, provider_name="cuda", input_dtype=np.float16
    )
    print(f"FP16 ONNX providers: {cuda_results['FP16 ONNX']['providers']}")

    print("Running INT8 ONNX on CUDA ...")
    cuda_results["INT8 ONNX"] = evaluate_onnx_model(
        int8_onnx_model, loader, provider_name="cuda", input_dtype=np.float32
    )
    print(f"INT8 ONNX providers: {cuda_results['INT8 ONNX']['providers']}")

    print_summary_table("CUDA Summary", cuda_results, model_sizes_mb)
    print_sample_predictions(cuda_results, label="INT8 ONNX")
else:
    print("\nCUDA is not available for both PyTorch and ONNX Runtime in this environment.")
    print("For Colab GPU, make sure:")
    print("1. Runtime type is set to GPU")
    print("2. onnxruntime-gpu is installed")

## 14. Recap

In this workshop you adapted pretrained AlexNet to MNIST, finetuned the classifier, compared accuracy and throughput against a pruned variant, exported both networks to ONNX, and checked that ONNX Runtime matches PyTorch on the same random input.

**Exported models** (paths follow your export cells, e.g. `../` relative to this notebook):

- **`alexnet_mnist.onnx`** — finetuned model weights. Use as the default deployment artifact.
- **`alexnet_mnist_fp16.onnx`** — converted FP16 finetuned model weights; Best for GPU.
- **`alexnet_mnist_int8_static.onnx`** — quantized INT8 finetuned model weights; Best for CPU.

You can run these in ONNX Runtime, TensorRT, OpenVINO, or any runtime that supports the ONNX opset you exported with.
